# N156 · SQL日期、留存与移动报表

**目标：** 对齐日期粒度后计算分母和滑窗。

**先修：** N155。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 日期运算；日期缺口；日聚合；首日/次日；分母；四舍五入；执行计划导览。

## 从问题到算法

日期不是普通相邻行。昨天要求自然日期相差一天，留存分母是独立玩家而不是登录行数。餐厅报表先按日期汇总，再按自然日范围求和：缺失日期不能把七行窗口伪装成七天。本章缺失日期贡献0，只在实际有消费的日期输出已覆盖完整七天的窗口。

## 最小实现

**本章接口：** `query_rising_temperature.sql`、`query_next_day_retention.sql`、`query_restaurant_growth.sql`

In [1]:
import sqlite3

SQL_QUERIES = {
    'query_rising_temperature.sql': r'''SELECT today.id
FROM Weather AS today JOIN Weather AS yesterday
ON yesterday.recordDate = DATE(today.recordDate, '-1 day')
WHERE today.temperature > yesterday.temperature;''',
    'query_next_day_retention.sql': r'''WITH first_login AS (
    SELECT player_id, MIN(event_date) AS first_date FROM Activity GROUP BY player_id
)
SELECT ROUND(COALESCE(1.0 * SUM(EXISTS (
    SELECT 1 FROM Activity AS a
    WHERE a.player_id = f.player_id AND a.event_date = DATE(f.first_date, '+1 day')
)) / COUNT(*), 0.0), 2) AS fraction
FROM first_login AS f;''',
    'query_restaurant_growth.sql': r'''WITH daily AS (
    SELECT visited_on, SUM(amount) AS daily_amount FROM Customer GROUP BY visited_on
), rolling AS (
    SELECT visited_on,
           SUM(daily_amount) OVER (
               ORDER BY JULIANDAY(visited_on)
               RANGE BETWEEN 6 PRECEDING AND CURRENT ROW
           ) AS amount
    FROM daily
)
SELECT visited_on, amount, ROUND(amount / 7.0, 2) AS average_amount
FROM rolling
WHERE JULIANDAY(visited_on) - (SELECT JULIANDAY(MIN(visited_on)) FROM daily) >= 6
ORDER BY visited_on;''',
}

def create_sqlite_fixture():
    connection=sqlite3.connect(':memory:')
    connection.executescript(r'''
CREATE TABLE Weather(id INTEGER PRIMARY KEY,recordDate TEXT,temperature INTEGER);
INSERT INTO Weather VALUES (1,'2020-12-31',10),(2,'2021-01-01',11),(3,'2021-01-03',20);
CREATE TABLE Activity(player_id INTEGER,device_id INTEGER,event_date TEXT,games_played INTEGER);
INSERT INTO Activity VALUES (1,1,'2020-12-31',1),(1,1,'2021-01-01',1),(1,2,'2021-01-01',2),
(2,1,'2021-01-01',1),(2,1,'2021-01-03',1),(3,1,'2021-01-02',1);
CREATE TABLE Customer(customer_id INTEGER,name TEXT,visited_on TEXT,amount INTEGER);
INSERT INTO Customer VALUES (1,'A','2021-01-01',10),(2,'B','2021-01-01',20),
(3,'C','2021-01-03',30),(4,'D','2021-01-07',70),(5,'E','2021-01-08',80);
''')
    return connection

def query_rows(connection,name):
    return connection.execute(SQL_QUERIES[name]).fetchall()

## 正确性、前提与复杂度

留存先按玩家求首日，再用EXISTS判断恰好次日是否出现，重复登录不重复计分。餐厅RANGE以JULIANDAY数值排序，6 PRECEDING到CURRENT ROW对应当前日及前六个自然日；与ROWS基于行数不同。分母固定7，空活动集合的0.0是本章明确的教学扩展。

**代价：** 先聚合日/玩家通常需分组与排序；窗口排序O(D log D)级，实际查询计划决定具体实现。DATE、JULIANDAY及此窗口写法已在SQLite执行，未宣称MySQL/PostgreSQL方言已测。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

db=create_sqlite_fixture()
for name in SQL_QUERIES:
    cursor=db.execute(SQL_QUERIES[name]); print(name); show_table([x[0] for x in cursor.description],cursor.fetchall())
print('实际SQLite版本',sqlite3.sqlite_version)
db.close()

query_rising_temperature.sql


id
2


query_next_day_retention.sql


fraction
0.33


query_restaurant_growth.sql


visited_on,amount,average_amount
2021-01-07,130,18.57
2021-01-08,180,25.71


实际SQLite版本 3.45.1


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
db=create_sqlite_fixture()
assert query_rows(db,'query_rising_temperature.sql')==[(2,)]
assert query_rows(db,'query_next_day_retention.sql')==[(0.33,)]
assert query_rows(db,'query_restaurant_growth.sql')==[('2021-01-07',130,18.57),('2021-01-08',180,25.71)]
# 重复同日记录不会增加玩家分母或留存次数。
db.execute("INSERT INTO Activity VALUES(1,3,'2021-01-01',99)")
assert query_rows(db,'query_next_day_retention.sql')==[(0.33,)]
db.execute('DELETE FROM Activity'); assert query_rows(db,'query_next_day_retention.sql')==[(0.0,)]
db.execute('DELETE FROM Customer'); assert query_rows(db,'query_restaurant_growth.sql')==[]
db.close()
print('N156: 所有本章断言通过')

N156: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 说明上一条记录不一定是昨天。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较SQLite教学SQL和题目MySQL方言，分别标注。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 197. Rising Temperature（canonical）
- 550. Game Play Analysis IV（canonical）
- 1321. Restaurant Growth（canonical）
- 262. Trips and Users（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。